In [0]:
dbutils.widgets.text('source_system','')
dbutils.widgets.text('table_name','')
dbutils.widgets.text('year','')
dbutils.widgets.text('month','')
dbutils.widgets.text('date','')
dbutils.widgets.text('landing_timestamp','')

source_system = dbutils.widgets.get('source_system')
table_name = dbutils.widgets.get('table_name')
landing_timestamp = dbutils.widgets.get('landing_timestamp')

YearMonthDatepath = dbutils.widgets.get('year') + '/' + dbutils.widgets.get('month') + '/' + dbutils.widgets.get('date')+'/'
landing_path = '/Volumes/man_catlog/landing/landing_volume/' + source_system + '/' + table_name + '/' + YearMonthDatepath

In [0]:
dbutils.fs.ls(landing_path)
print(landing_path)

In [0]:
df = spark.read.format('parquet')\
                .load(landing_path)

In [0]:
display(df.limit(3))

In [0]:
from pyspark.sql import functions as F

updated_df = df.withColumn('landing_timestamp',F.lit(landing_timestamp))\
               .withColumn('insert_timeStamp',F.current_timestamp())

In [0]:
display(updated_df.limit(5))

In [0]:
updated_df.write.mode("append")\
            .option("path", f"abfss://ext-adls-unity-catalog@storageacclpv0.dfs.core.windows.net/caresync_schema_bronze/{table_name}")\
            .saveAsTable(f"man_catlog.caresync_schema_bronze.{table_name}")

In [0]:
%sql
SELECT * FROM IDENTIFIER('man_catlog.caresync_schema_bronze.' || :table_name) limmit 5